In [1]:
# installed arxiv for llm to work on mainly research paper
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

C:\Users\moham\AppData\Local\Temp\ipykernel_27692\471557318.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [2]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1 ,doc_content_chars_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

In [3]:
wiki.name

'wikipedia'

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

embeddings = OllamaEmbeddings(model="nomic-embed-text")

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents,embeddings)
retriever = vectordb.as_retriever()
retriever

USER_AGENT environment variable not set, consider setting it to identify your requests.


VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001924D34AA50>, search_kwargs={})

In [5]:
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for information about LangSmith. For any questions about LangSmith, you must use this tool."
)

In [6]:
retriever_tool.name

'langsmith_search'

In [7]:
#from langchain_community.utilities import ArxivAPIWrapper
#from langchain_community.tools import ArxivQueryRun

#arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
#arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
 #arxiv version = 1.4.8

#Claude code
import re
import arxiv
from langchain_core.tools import tool

_ARXIV_ID_RE = re.compile(r"^\d{4}\.\d{4,5}(v\d+)?$")

@tool
def arxiv_search(query: str) -> str:
    """Search Arxiv.org for scientific papers (Physics, Math, CS, Stats, etc.).
    Input can be a plain text search query or an arXiv ID like '1605.08386'."""
    client = arxiv.Client()
    if _ARXIV_ID_RE.match(query.strip()):
        search = arxiv.Search(id_list=[query.strip()], max_results=1)
    else:
        search = arxiv.Search(query=query, max_results=1)

    results = list(client.results(search))
    if not results:
        return "No results found on Arxiv for that query."

    r = results[0]
    return (
        f"Published: {r.published.date()}\n"
        f"Title: {r.title}\n"
        f"Authors: {', '.join(a.name for a in r.authors)}\n"
        f"Summary: {r.summary[:200]}"
    )


In [8]:
tools = [wiki,arxiv_search,retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\moham\\Desktop\\RAG pipeline\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='arxiv_search', description="Search Arxiv.org for scientific papers (Physics, Math, CS, Stats, etc.).\nInput can be a plain text search query or an arXiv ID like '1605.08386'.", args_schema=<class 'langchain_core.utils.pydantic.arxiv_search'>, func=<function arxiv_search at 0x00000192328B7950>),
 StructuredTool(name='langsmith_search', description='Search for information about LangSmith. For any questions about LangSmith, you must use this tool.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001924D33AA30>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001927D88E560>)]

In [9]:
from dotenv import load_dotenv
load_dotenv()
import os
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3",
    temperature=0
)

In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful AI assistant."
)



In [12]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Who is Soldier  boy from comics?"}]}
)

print(response["messages"][-1].content)

Soldier Boy refers to three distinct superhero characters in the comic book series *Herogasm* and *The Boys*, created by writer Garth Ennis and artist John McCrea. These characters are part of the larger "Herogasm" storyline, which explores themes of superhero culture and exploitation. The term "Soldier Boy" is used to describe different characters within these comics, though specific details about each variant would require further clarification or context from the user.
